# Chapter 5.2 MC 제어와 블랙잭 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter05_2_mc_control_blackjack.ipynb)

책 본문: [Chapter 5](https://smhanlab.com/book-ml/kor/ml2/chapter05.html)

이 노트북에서는 블랙잭에서 "카드를 더 받을지(hit) 멈출지(stand)"를
첫방문 MC 제어(\(\varepsilon\)-greedy 탐험, \(\gamma=1\))로 학습하고,
동적계획법(가치반복)으로 구한 정확한 \(Q^*\)와 숫자로 비교합니다.
블랙잭은 한 판이 분명히 끝나는 **유한 호라이즌** 과업이므로
\(\gamma=1\), 보상 \(-1/0/+1\)을 씁니다(소프트 에이스는 무시).

## 1. 블랙잭 환경 (단순화 버전)

- **상태** \(s = (\text{플레이어 합}, \text{딜러의 보이는 카드})\),
  학습 대상은 플레이어 합 12~21 (12 미만은 무조건 히트라 학습의 여지가 없음)
- **행동**: 0 = **스틱**(stand, 멈추기), 1 = **히트**(hit, 카드 더 받기)
- **히트**: 무한 카드덱(1덱의 카드 분포: 2~9 각각 4/52, 10·J·Q·K 합쳐 16/52,
  A 4/52)에서 한 장 추가. 합이 21을 넘으면 **버스트** → 보상 \(-1\), 종료
- **스틱**: 딜러가 합이 17 이상일 때까지 카드를 받고(딜러가 버스트하면
  플레이어 승리), 최종 합으로 승패 — \(+1\)(이김) / 0(무승부) / \(-1\)(지면)

In [1]:
import random

DECK = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]  # 13장: 10 계열이 4/13, 나머지는 각 1/13

def draw_card():
    return random.choice(DECK)

def step(s, a):
    """s=(플레이어 합, 딜러 보이는 카드), a: 0=스틱, 1=히트.
    (다음 상태, 보상, done) 반환. 스틱하면 딜러 플레이를 한꺼번에 시뮬레이션."""
    p, d = s
    if a == 1:  # 히트
        p += draw_card()
        if p > 21:                       # 버스트
            return (p, d), -1.0, True
        return (p, d), 0.0, False
    # 스틱: 딜러가 합 17 이상까지 카드 받기
    ds = d
    while ds < 17:
        ds += draw_card()
    if ds > 21 or p > ds:                # 딜러 버스트, 또는 플레이어 합이 높음
        r = 1.0
    elif p == ds:
        r = 0.0
    else:
        r = -1.0
    return (p, d), r, True

In [2]:
# 환경 동작 확인: 완전 무작위(epsilon=1)로 5판만 돌려보기
random.seed(0)
for i in range(5):
    s = (draw_card() + draw_card(), draw_card())   # 플레이어 2장, 딜러 1장
    while True:
        a = random.randrange(2)
        s, r, done = step(s, a)
        if done:
            break
    print(f"에피소드 {i+1}: 최종 상태={s}, 보상={r:+.0f}")

에피소드 1: 최종 상태=(19, 8), 보상=+1
에피소드 2: 최종 상태=(26, 11), 보상=-1
에피소드 3: 최종 상태=(17, 5), 보상=+1
에피소드 4: 최종 상태=(23, 11), 보상=-1
에피소드 5: 최종 상태=(21, 10), 보상=+1


## 2. MC 제어 (첫방문 + 증분 평균)

본문 `mc_control`과 같은 뼈대입니다 — \(\varepsilon\)-greedy로 에피소드를
실행한 뒤, 에피소드를 **뒤에서부터** 훑으며 리턴 \(G\)를 재귀적으로 계산하고,
각 (상태, 행동) 쌍은 **에피소드 내 첫 방문**만 증분 평균으로 갱신합니다.
\(\gamma=1\), 플레이어 합 12 이상인 상태만 학습 대상입니다.

In [3]:
def gen_episode(Q, epsilon):
    """\varepsilon-greedy(미방문 상태는 무작위)로 한 판(에피소드) 생성."""
    s = (draw_card() + draw_card(), draw_card())
    episode = []
    while True:
        if s[0] < 12:
            a = 1                                        # 12 미만: 무조건 히트
        elif s not in Q or random.random() < epsilon:
            a = random.randrange(2)                      # 미방문 상태 또는 탐험
        else:
            a = max(range(2), key=lambda x: Q[s][x])     # 그리는 행동
        ns, r, done = step(s, a)
        episode.append((s, a, r))
        if done:
            break
        s = ns
    return episode

def mc_update(Q, counts, episode, gamma=1.0):
    """첫방문 MC 갱신: 증분 평균(Chapter 2)으로 Q[s][a]를 업데이트."""
    G, visited = 0.0, set()
    for s, a, r in reversed(episode):
        G = r + gamma * G
        if s[0] >= 12 and (s, a) not in visited:         # 첫방문
            visited.add((s, a))
            if s not in Q:
                Q[s] = [0.0, 0.0]
                counts[s] = [0, 0]
            counts[s][a] += 1
            Q[s][a] += (G - Q[s][a]) / counts[s][a]

random.seed(0)
Q, counts = {}, {}
N1 = 300_000
for _ in range(N1):
    mc_update(Q, counts, gen_episode(Q, epsilon=0.1))
print(f"{N1:,} 에피소드 완료 (epsilon=0.1). 학습한 상태 수: {len(Q)}")

300,000 에피소드 완료 (epsilon=0.1). 학습한 상태 수: 110


In [4]:
# 본문과 같은 표본: 20 근처는 스틱, 12에서 딜러 10이면 히트
for s in [(20, 3), (20, 10), (12, 10)]:
    q = Q[s]
    best = "스틱" if q[0] >= q[1] else "히트"
    print(f"상태(합={s[0]}, 딜러={s[1]}):  Q(스틱)={q[0]:+.3f}, Q(히트)={q[1]:+.3f} -> {best}")

상태(합=20, 딜러=3):  Q(스틱)=+0.682, Q(히트)=-1.000 -> 스틱
상태(합=20, 딜러=10):  Q(스틱)=+0.443, Q(히트)=-1.000 -> 스틱
상태(합=12, 딜러=10):  Q(스틱)=-0.520, Q(히트)=-0.470 -> 히트


## 3. 정확한 \(Q^*\)(동적계획법)과 비교

이 절의 환경은 전이가 **무작위 카드 한 장의 분포**에만 의존하므로,
시뮬레이션 없이 정확한 벨만백업(= 가치반복)을 직접 계산할 수 있습니다:

- **스틱**: 딜러의 카드 뽑기 과정은 유한한 재귀 → 기대 보상 closed form
- **히트**: \(Q^*(s,\text{히트}) = \mathbb{E}_c[-1 \cdot \mathbb{1}[p+c>21]
  + \mathbb{1}[p+c\le 21] \cdot \max_a Q^*(s,a)]\) — 버스트하지 않는
  히트는 같은 상태로 돌아와 다시 최적 행동을 하므로

In [5]:
import functools
from collections import Counter

CARD_CNT = Counter(DECK)  # {2:1, ..., 9:1, 10:4, 11:1}

def card_exp(f):
    """무한 덱에서 카드 한 장을 뽑아 f(카드)의 기대."""
    return sum(cnt * f(v) for v, cnt in CARD_CNT.items()) / 13.0

@functools.lru_cache(maxsize=None)
def stand_value(p, d):
    """플레이어(합 p)가 스틱했을 때, 딜러의 현재 합 d에서 시작하는 기대 보상.
    딜러는 합 17 이상까지 뽑고, 21 초과(버스트)하면 플레이어가 이김."""
    if d > 21:
        return 1.0
    if d >= 17:
        return 1.0 if p > d else (0.0 if p == d else -1.0)
    return card_exp(lambda c: stand_value(p, d + c))

Qstar = {}
for p in range(12, 22):
    for d in range(2, 13):
        s = (p, d)
        q_stand = stand_value(p, d)
        q_hit = 0.0
        for _ in range(200):   # 고정점 반복: 같은 상태의 max(스틱, 히트)에 수렴
            q_hit = card_exp(lambda c: -1.0 if p + c > 21 else max(q_stand, q_hit))
        Qstar[s] = (q_stand, q_hit)
print("정확한 Q* 계산 완료:", len(Qstar), "개 상태")

# 학습 정책 vs 정확 정책의 일치율 (전체 학습 상태)
agree = sum(
    int((Q[s][0] >= Q[s][1]) == (Qstar[s][0] >= Qstar[s][1]))
    for s in Qstar if s in Q
)
total = sum(1 for s in Qstar if s in Q)
print(f"정책 일치율: {agree}/{total}")

정확한 Q* 계산 완료: 110 개 상태
정책 일치율: 88/100


In [6]:
# 본문 비교표와 같은 표본 상태: 학습 Q vs 정확 Q*
rows = [(20, 3), (20, 10), (12, 10), (12, 2), (15, 10), (15, 2)]
print(f"{'상태':>10} {'학습Q(스틱)':>12} {'학습Q(히트)':>12} {'Q*(스틱)':>10} {'Q*(히트)':>10} {'최적행동':>8}")
for s in rows:
    qs, qh = Q[s]
    ts, th = Qstar[s]
    act = "스틱" if ts >= th else "히트"
    print(f"({s[0]:>2}, {s[1]:>2})  {qs:>+10.3f} {qh:>+10.3f} {ts:>+8.3f} {th:>+8.3f} {act:>6}")

        상태      학습Q(스틱)      학습Q(히트)     Q*(스틱)     Q*(히트)     최적행동
(20,  3)      +0.682     -1.000   +0.680   -1.000     스틱
(20, 10)      +0.443     -1.000   +0.441   -1.000     스틱
(12, 10)      -0.520     -0.470   -0.540   -0.717     스틱
(12,  2)      -0.227     -0.276   -0.173   -0.491     스틱
(15, 10)      -0.543     -0.643   -0.540   -0.823     스틱
(15,  2)      -0.167     -0.500   -0.173   -0.682     스틱


## 4. \(\varepsilon\) 감소(탐험 스케줄)

\(\varepsilon=0.1\)을 30만 에피소드까지 유지하면 \(Q\)가 계속 흔들려
경계 상태의 판단이 굳어지기 어렵습니다. 실전처럼 학습이 진행될수록
\(\varepsilon\)를 줄여(ε-decay) — 여기서 10만 에피소드를
\(\varepsilon=0.02\)로 추가 학습 — 하면 정확 정책과의 일치율이
조금 올라가는 것을 볼 수 있습니다(총 40만 에피소드).

In [7]:
def agreement(Q):
    a = sum(int((Q[s][0] >= Q[s][1]) == (Qstar[s][0] >= Qstar[s][1])) for s in Qstar if s in Q)
    t = sum(1 for s in Qstar if s in Q)
    return a, t

def q_err(Q):
    e = [abs(Q[s][a] - Qstar[s][a]) for s in Q for a in range(2) if s in Qstar]
    return sum(e) / len(e), max(e)

a, t = agreement(Q)
me, xe = q_err(Q)
print(f"감소 전: 일치 {a}/{t},  평균 |Q-Q*|={me:.3f},  최대={xe:.3f}")

N2 = 100_000
for _ in range(N2):
    mc_update(Q, counts, gen_episode(Q, epsilon=0.02))
a, t = agreement(Q)
me, xe = q_err(Q)
print(f"감소 후: 일치 {a}/{t},  평균 |Q-Q*|={me:.3f},  최대={xe:.3f}")
print(f"총 에피소드: {N1 + N2:,}")

감소 전: 일치 88/100,  평균 |Q-Q*|=0.080,  최대=0.398


감소 후: 일치 88/100,  평균 |Q-Q*|=0.080,  최대=0.388
총 에피소드: 400,000


In [8]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
import numpy as np
from matplotlib.patches import Rectangle

IMG = "/home/smhan/book-ml/kor/src/images"

rows = list(range(21, 11, -1))   # 플레이어 합 21 -> 12 (위 -> 아래)
cols = list(range(2, 11)) + [11] # 딜러 2 -> A (왼쪽 -> 오른쪽)
col_lbl = [str(v) for v in range(2, 11)] + ["A"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5),
                         gridspec_kw={"width_ratios": [1.25, 1]})

# --- 좌: 학습한 그리디 정책 그리드 ---
ax = axes[0]
for i, p in enumerate(rows):
    for j, d in enumerate(cols):
        s = (p, d)
        ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1,
                               facecolor="#f2f2f2", edgecolor="white", lw=1.5))
        if s in Q:
            a_mc = 0 if Q[s][0] >= Q[s][1] else 1
            a_star = 0 if Qstar[s][0] >= Qstar[s][1] else 1
            txt = "S" if a_mc == 0 else "H"
            color = "tab:green" if a_mc == 0 else "tab:red"
            if a_mc != a_star:
                txt += "*"
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=12, color=color, weight="bold")
ax.set_xlim(-0.5, len(cols) - 0.5)
ax.set_ylim(len(rows) - 0.5, -0.5)
ax.set_xticks(range(len(cols)), col_lbl)
ax.set_yticks(range(len(rows)), rows)
ax.set_xlabel("딜러의 오픈 카드")
ax.set_ylabel("플레이어 합")
ax.set_title("학습한 그리디 정책 (S=스틱, H=히트, *=정확한 정책과 다름)")

# --- 우: Q(스틱) - Q(히트) 차이 ---
ax2 = axes[1]
diff = np.array([[Q[(p, d)][0] - Q[(p, d)][1] for d in cols] for p in rows])
im = ax2.imshow(diff, aspect="auto", cmap="coolwarm", vmin=-1.5, vmax=1.5)
ax2.set_xticks(range(len(cols)), col_lbl)
ax2.set_yticks(range(len(rows)), rows)
ax2.set_xlabel("딜러의 오픈 카드")
ax2.set_ylabel("플레이어 합")
ax2.set_title("Q(스틱) − Q(히트)  (양수=스틱 유리)")
fig.colorbar(im, ax=ax2, label="Q(스틱) − Q(히트)")

fig.tight_layout()
fig.savefig(IMG + "/ch05_2_blackjack_strategy.svg", bbox_inches="tight")
import base64
from IPython.display import HTML, display
with open(IMG + "/ch05_2_blackjack_strategy.svg", "rb") as _f:
    _b64 = base64.b64encode(_f.read()).decode()
display(HTML('<img src="data:image/svg+xml;base64,' + _b64 + '" style="width:900px"/>'))
print("저장:", IMG + "/ch05_2_blackjack_strategy.svg")

저장: /home/smhan/book-ml/kor/src/images/ch05_2_blackjack_strategy.svg


## 5. 정리

- **전이 계산 한 줄 없이** 40만 에피소드의 시뮬레이션만으로 블랙잭 **기본 전략의
  골격**을 재현했습니다 — 17 이상은 전부 스틱, 12~13은 전부 히트, 그 사이는
  딜러 카드에 따라 갈리는 대각선 패턴.
- 학습된 \(Q\)는 \(Q^*\)가 아니라 **\(\varepsilon\)-greedy 행동 정책의
  \(Q^{\pi}\)**입니다 — 탐험으로 섞인 무작위 행동의 낮은 보상이 반영되어
  \(Q^*\)보다 약간 낮게 측정되고, \(\varepsilon\)를 줄이면
  \(Q^{\pi} \to Q^*\)에 가까워집니다.
- **방문 횟수가 적은 (상태, 행동) 쌍의 Q는 노이즈가 많습니다** — "방문 횟수 =
  신뢰도"라는 직감은 5.3의 중요도 샘플링, Chapter 6의 TD, Chapter 9의 DQN까지
  공통으로 적용됩니다.